# 00. Từ điển dữ liệu

**Mục tiêu:** Mô tả 13 bảng: mỗi dòng là gì, cột nào nghĩa là gì, kiểu dữ liệu, % trống, khoá nối giữa các bảng có khớp không.

**Bảng dữ liệu dùng:** 13 bảng trong `data/raw/datathon-2026-round-1/` (bỏ `sample_submission`)

In [1]:
import sys
from pathlib import Path

# Tìm thư mục gốc repo (chứa src/ds317) để import thư viện chung
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "ds317").is_dir())
sys.path.insert(0, str(ROOT / "src"))

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from ds317 import load, load_many, order_lines, revenue_daily, weekly_category, REVENUE_LABELS
from ds317.viz import setup_style, nb_out, save_fig, source_note, key_numbers, MILLION, UNIT, NOTE_SALES, NOTE_TRAFFIC

%matplotlib inline
warnings.filterwarnings("ignore", category=FutureWarning)
setup_style()
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
FIG, TAB = nb_out("00_tu_dien_du_lieu")
print("Ảnh lưu tại:", FIG.relative_to(ROOT))

Ảnh lưu tại: outputs\eda\00_tu_dien_du_lieu


Phần **kỹ thuật** (kiểu, % trống, min/max, ví dụ, khoá) do code tính. Phần **ý nghĩa cột** lấy từ
`src/ds317/dictionary.py` (mô tả khách quan, đã đối chiếu giá trị thật) – sửa ở file đó nếu cần, chạy lại notebook để cập nhật.
Mục **"3 điều lạ"** dưới mỗi bảng để nhóm điền.

In [2]:
from ds317.data import TABLES
from ds317.profile import profile_table, key_coverage, duplicate_keys, date_range
from ds317.dictionary import TABLE_DESC, COLUMN_DESC

NAMES = [t for t in TABLES if t != "sample_submission"]
t = load_many(*NAMES)

## Mỗi bảng lớn cỡ nào, phủ khoảng thời gian nào?

In [3]:
rows = []
for name in NAMES:
    df = t[name]
    lo, hi = date_range(df, name)
    rows.append({"bang": name, "so_dong": len(df), "so_cot": df.shape[1],
                 "tu_ngay": lo.date() if lo is not None else "", "den_ngay": hi.date() if hi is not None else "",
                 "pct_o_trong": round(df.isna().to_numpy().mean() * 100, 2),
                 "mot_dong_la": TABLE_DESC[name][0], "khoa_chinh": TABLE_DESC[name][1]})
overview = pd.DataFrame(rows)
overview.to_csv(TAB / "overview.csv", index=False, encoding="utf-8-sig")
overview

,bang,so_dong,so_cot,tu_ngay,den_ngay,pct_o_trong,mot_dong_la,khoa_chinh
0,sales,3833,3,2012-07-04,2022-12-31,0.00,1 ngày (2012-07-04 → 2022-12-31): doanh thu và...,Date
1,web_traffic,3652,7,2013-01-01,2022-12-31,0.00,1 ngày (2013-01-01 → 2022-12-31): chỉ số truy ...,date
2,orders,646945,8,2012-07-04,2022-12-31,0.00,1 đơn hàng,order_id
3,order_items,714669,7,,,23.04,1 sản phẩm trong 1 đơn (1 đơn có 1–5 dòng),"(order_id, product_id) – có 16 dòng trùng"
4,products,2412,8,,,0.00,1 SKU (sản phẩm × size × màu),product_id
5,customers,121930,7,2012-01-17,2022-12-31,0.00,1 khách hàng,customer_id
6,geography,39948,4,,,0.00,1 mã bưu chính (zip),zip
7,payments,646945,4,,,0.00,1 lần thanh toán của 1 đơn (mỗi đơn đúng 1 dòng),order_id
8,shipments,566067,4,2012-07-04,2022-12-31,0.00,1 lần giao hàng của 1 đơn (chỉ đơn đã giao/đan...,order_id
9,returns,39939,7,2012-07-11,2022-12-31,0.00,1 lần trả 1 sản phẩm của 1 đơn,return_id


In [4]:
key_numbers({
    "Tổng số dòng 13 bảng": f"{overview.so_dong.sum():,}",
    "Bảng lớn nhất": f"{overview.loc[overview.so_dong.idxmax(), 'bang']} ({overview.so_dong.max():,} dòng)",
    "Bảng có ô trống": ", ".join(overview.loc[overview.pct_o_trong > 0, "bang"]),
    "Khoảng ngày sales": f"{overview.set_index('bang').loc['sales', 'tu_ngay']} → {overview.set_index('bang').loc['sales', 'den_ngay']}",
})

Số liệu chính:
  - Tổng số dòng 13 bảng: 2,960,188
  - Bảng lớn nhất: order_items (714,669 dòng)
  - Bảng có ô trống: order_items, promotions
  - Khoảng ngày sales: 2012-07-04 → 2022-12-31


## Các bảng nối với nhau qua khoá nào, khoá có khớp không?

In [5]:
PAIRS = [
    ("order_items", "orders", "order_id"), ("order_items", "products", "product_id"),
    ("orders", "customers", "customer_id"), ("customers", "geography", "zip"), ("orders", "geography", "zip"),
    ("payments", "orders", "order_id"), ("shipments", "orders", "order_id"),
    ("returns", "orders", "order_id"), ("reviews", "orders", "order_id"),
    ("inventory", "products", "product_id"),
]
cov = pd.DataFrame([key_coverage(t[c], t[p], k, c, p) for c, p, k in PAIRS])
# chiều ngược lại: bao nhiêu đơn có payments / shipments / returns / reviews
rev = pd.DataFrame([{**key_coverage(t["orders"], t[c], "order_id", "orders", c), "ghi_chu": f"% đơn có bản ghi {c}"}
                    for c in ["payments", "shipments", "returns", "reviews"]])
cov = pd.concat([cov, rev], ignore_index=True)
cov.to_csv(TAB / "key_coverage.csv", index=False, encoding="utf-8-sig")
cov

,bang_con,bang_cha,khoa,so_khoa_con,so_khoa_thieu_o_cha,pct_khop,ghi_chu
0,order_items,orders,order_id,646945,0,100.0000,NaN
1,order_items,products,product_id,1598,0,100.0000,NaN
2,orders,customers,customer_id,90246,0,100.0000,NaN
3,customers,geography,zip,31491,0,100.0000,NaN
4,orders,geography,zip,29932,0,100.0000,NaN
5,payments,orders,order_id,646945,0,100.0000,NaN
6,shipments,orders,order_id,566067,0,100.0000,NaN
7,returns,orders,order_id,36062,0,100.0000,NaN
8,reviews,orders,order_id,111369,0,100.0000,NaN
9,inventory,products,product_id,1624,0,100.0000,NaN


In [6]:
dups = pd.DataFrame([
    {"bang": n, "khoa": ", ".join(k), "so_dong_trung": duplicate_keys(t[n], k)}
    for n, k in [("orders", ["order_id"]), ("products", ["product_id"]), ("customers", ["customer_id"]),
                 ("sales", ["Date"]), ("geography", ["zip"]), ("payments", ["order_id"]), ("shipments", ["order_id"]),
                 ("order_items", ["order_id", "product_id"]), ("inventory", ["snapshot_date", "product_id"])]
])
dups.to_csv(TAB / "duplicate_keys.csv", index=False, encoding="utf-8-sig")
dups

,bang,khoa,so_dong_trung
0,orders,order_id,0
1,products,product_id,0
2,customers,customer_id,0
3,sales,Date,0
4,geography,zip,0
5,payments,order_id,0
6,shipments,order_id,0
7,order_items,"order_id, product_id",16
8,inventory,"snapshot_date, product_id",0


In [7]:
key_numbers({
    "Cặp khoá con→cha khớp 100%": f"{(cov.iloc[:len(PAIRS)].pct_khop == 100).sum()}/{len(PAIRS)}",
    "% đơn có shipments": f"{cov.set_index('bang_cha').loc['shipments', 'pct_khop']:.2f}%",
    "% đơn có reviews": f"{cov.set_index('bang_cha').loc['reviews', 'pct_khop']:.2f}%",
    "Khoá chính bị trùng": ", ".join(f"{r.bang} ({r.so_dong_trung})" for r in dups.itertuples() if r.so_dong_trung) or "không có",
})

Số liệu chính:
  - Cặp khoá con→cha khớp 100%: 10/10
  - % đơn có shipments: 87.50%
  - % đơn có reviews: 17.21%
  - Khoá chính bị trùng: order_items (16)


> **Đọc chart (nhóm điền):**
> 1. Kết luận 1 câu có số: …
> 2. Số kiểm tra lại từ data: …
> 3. Điều lạ / chưa giải thích được: …
> 4. Ảnh hưởng đến tiền xử lý hoặc mô hình: …

## Từ điển từng bảng

Mỗi bảng: ô mô tả (1 dòng là gì, khoá) → bảng profile (cột, ý nghĩa, kiểu, % trống, min/max, ví dụ; lưu
`tables/profile_<bảng>.csv`) → ô **3 điều lạ** để nhóm điền.

In [8]:
def show_profile(name):
    p = profile_table(t[name], name)
    p.insert(2, "y_nghia", p["cot"].map(COLUMN_DESC.get(name, {})).fillna(""))
    p.to_csv(TAB / f"profile_{name}.csv", index=False, encoding="utf-8-sig")
    return p.drop(columns="bang").style.hide(axis="index").set_properties(**{"text-align": "left"})

### Từ điển bảng `sales`

- **1 dòng là:** 1 ngày (2012-07-04 → 2022-12-31): doanh thu và giá vốn tổng của cả công ty
- **Khoá chính:** Date
- **Khoá nối:** Date ↔ orders.order_date (cộng order_items theo ngày)

In [9]:
show_profile("sales")

cot,y_nghia,kieu,pct_trong,so_gia_tri_khac_nhau,min,max,vi_du
Date,Ngày,datetime64[us],0.000000,3833,2012-07-04,2022-12-31,2012-07-04 | 2012-07-05 | 2012-07-06
Revenue,"Doanh thu ngày = SUM(quantity × unit_price) của mọi đơn đặt trong ngày, gồm đơn huỷ, trước chiết khấu",float64,0.000000,3833,2.798e+05,2.091e+07,5.124e+06 | 2.752e+06 | 3.054e+06
COGS,Giá vốn ngày = SUM(quantity × products.cogs) của mọi đơn đặt trong ngày,float64,0.000000,3833,2.366e+05,1.654e+07,3.983e+06 | 2.151e+06 | 2.518e+06


> **3 điều lạ của bảng `sales` (nhóm điền):**
> 1. …
> 2. …
> 3. …

### Từ điển bảng `web_traffic`

- **1 dòng là:** 1 ngày (2013-01-01 → 2022-12-31): chỉ số truy cập website
- **Khoá chính:** date
- **Khoá nối:** date ↔ sales.Date

In [10]:
show_profile("web_traffic")

cot,y_nghia,kieu,pct_trong,so_gia_tri_khac_nhau,min,max,vi_du
date,Ngày,datetime64[us],0.000000,3652,2013-01-01,2022-12-31,2013-01-01 | 2013-01-02 | 2013-01-03
sessions,Số phiên truy cập trong ngày,int64,0.000000,3447,7973,50947,9760 | 10456 | 10076
unique_visitors,Số người truy cập khác nhau trong ngày,int64,0.000000,3382,6136,40430,7253 | 8151 | 7458
page_views,Số lượt xem trang,int64,0.000000,3620,30451,275560,39093 | 47611 | 36963
bounce_rate,"Tỷ lệ thoát (dạng tỷ số, khoảng 0.003–0.006)",float64,0.000000,261,0.0032,0.0058,0.00514 | 0.00406 | 0.00401
avg_session_duration_sec,Thời lượng phiên trung bình (giây),float64,0.000000,1771,100.1,319.9,102.9 | 120.5 | 263.6
traffic_source,"Nguồn truy cập ghi cho ngày đó (1 giá trị/ngày): organic_search, paid_search, social_media, email_campaign, referral, direct",str,0.000000,6,,,organic_search | direct | referral


> **3 điều lạ của bảng `web_traffic` (nhóm điền):**
> 1. …
> 2. …
> 3. …

### Từ điển bảng `orders`

- **1 dòng là:** 1 đơn hàng
- **Khoá chính:** order_id
- **Khoá nối:** customer_id → customers; zip → geography

In [11]:
show_profile("orders")

cot,y_nghia,kieu,pct_trong,so_gia_tri_khac_nhau,min,max,vi_du
order_id,Mã đơn hàng,int64,0.000000,646945,1,834397,1 | 2 | 3
order_date,Ngày đặt hàng,datetime64[us],0.000000,3833,2012-07-04,2022-12-31,2012-07-04 | 2012-07-06 | 2012-07-05
customer_id,Mã khách đặt đơn,int64,0.000000,90246,1,157563,58578 | 58621 | 58811
zip,Mã bưu chính giao hàng,int64,0.000000,29932,1001,99950,1109 | 1330 | 1473
order_status,"Trạng thái đơn: created, paid, shipped, delivered, returned, cancelled",str,0.000000,6,,,delivered | returned | shipped
payment_method,"Phương thức thanh toán: credit_card, paypal, cod, apple_pay, bank_transfer",str,0.000000,5,,,credit_card | cod | paypal
device_type,"Thiết bị đặt hàng: mobile, desktop, tablet",str,0.000000,3,,,desktop | mobile | tablet
order_source,"Kênh dẫn tới đơn: organic_search, paid_search, social_media, email_campaign, referral, direct",str,0.000000,6,,,paid_search | direct | referral


> **3 điều lạ của bảng `orders` (nhóm điền):**
> 1. …
> 2. …
> 3. …

### Từ điển bảng `order_items`

- **1 dòng là:** 1 sản phẩm trong 1 đơn (1 đơn có 1–5 dòng)
- **Khoá chính:** (order_id, product_id) – có 16 dòng trùng
- **Khoá nối:** order_id → orders; product_id → products; promo_id, promo_id_2 → promotions

In [12]:
show_profile("order_items")

cot,y_nghia,kieu,pct_trong,so_gia_tri_khac_nhau,min,max,vi_du
order_id,Mã đơn,int64,0.000000,646945,1,834397,1 | 2 | 3
product_id,Mã sản phẩm (SKU),int64,0.000000,1598,1,2412,2400 | 609 | 396
quantity,Số lượng mua (1–8),int64,0.000000,8,1,8,7 | 3 | 5
unit_price,Đơn giá bán thực tế của dòng (khác products.price ở hầu hết dòng),float64,0.000000,501330,392.6,4.306e+04,"1,138 | 1.017e+04 | 1.122e+04"
discount_amount,Tiền chiết khấu của cả dòng (không phải mỗi đơn vị),float64,0.000000,204449,0,3.524e+04,"0 | 1,128 | 1,312"
promo_id,Mã khuyến mãi áp dụng (trống = không có khuyến mãi),str,61.340000,50,,,PROMO-0006 | PROMO-0001 | PROMO-0002
promo_id_2,Mã khuyến mãi thứ 2 khi cộng dồn (gần như luôn trống),str,99.970000,2,,,PROMO-0015 | PROMO-0025


> **3 điều lạ của bảng `order_items` (nhóm điền):**
> 1. …
> 2. …
> 3. …

### Từ điển bảng `products`

- **1 dòng là:** 1 SKU (sản phẩm × size × màu)
- **Khoá chính:** product_id
- **Khoá nối:** được tham chiếu bởi order_items, returns, reviews, inventory

In [13]:
show_profile("products")

cot,y_nghia,kieu,pct_trong,so_gia_tri_khac_nhau,min,max,vi_du
product_id,Mã SKU,int64,0.000000,2412,1,2412,536 | 537 | 538
product_name,"Tên sản phẩm (thương hiệu + mã mẫu, vd. SaigonFlex UC-01)",str,0.000000,2172,,,SaigonFlex UC-01 | SaigonFlex UC-02 | SaigonFlex UC-03
category,"Danh mục: Streetwear, Outdoor, Casual, GenZ (4 danh mục của biến mục tiêu)",str,0.000000,4,,,Streetwear | Casual | Outdoor
segment,"Phân khúc: Activewear, Everyday, Performance, Balanced, Standard, Premium, All-weather, Trendy",str,0.000000,8,,,Everyday | Performance | Balanced
size,"Cỡ: S, M, L, XL",str,0.000000,4,,,S | M | L
color,Màu (8 màu),str,0.000000,10,,,green | silver | pink
price,Giá niêm yết,float64,0.000000,1990,9.057,4.095e+04,"1.106e+04 | 9,523 | 1.595e+04"
cogs,Giá vốn 1 đơn vị,float64,0.000000,2381,5.184,3.89e+04,"9,705 | 5,394 | 1.137e+04"


> **3 điều lạ của bảng `products` (nhóm điền):**
> 1. …
> 2. …
> 3. …

### Từ điển bảng `customers`

- **1 dòng là:** 1 khách hàng
- **Khoá chính:** customer_id
- **Khoá nối:** zip → geography

In [14]:
show_profile("customers")

cot,y_nghia,kieu,pct_trong,so_gia_tri_khac_nhau,min,max,vi_du
customer_id,Mã khách hàng,int64,0.000000,121930,1,157563,1 | 2 | 3
zip,Mã bưu chính của khách,int64,0.000000,31491,1001,99950,15201 | 15202 | 15203
city,Thành phố,str,0.000000,42,,,Hai Phong | Phu Ly | Viet Tri
signup_date,Ngày tạo tài khoản,datetime64[us],0.000000,3941,2012-01-17,2022-12-31,2021-12-30 | 2013-12-27 | 2018-07-24
gender,"Giới tính: Female, Male, Non-binary",str,0.000000,3,,,Female | Male | Non-binary
age_group,"Nhóm tuổi: 18-24, 25-34, 35-44, 45-54, 55+",str,0.000000,5,,,35-44 | 45-54 | 18-24
acquisition_channel,"Kênh thu hút khách: organic_search, social_media, paid_search, email_campaign, referral, direct",str,0.000000,6,,,social_media | email_campaign | organic_search


> **3 điều lạ của bảng `customers` (nhóm điền):**
> 1. …
> 2. …
> 3. …

### Từ điển bảng `geography`

- **1 dòng là:** 1 mã bưu chính (zip)
- **Khoá chính:** zip
- **Khoá nối:** được tham chiếu bởi customers, orders

In [15]:
show_profile("geography")

cot,y_nghia,kieu,pct_trong,so_gia_tri_khac_nhau,min,max,vi_du
zip,Mã bưu chính,int64,0.000000,39948,1,99950,15201 | 15202 | 15203
city,Thành phố (42 giá trị),str,0.000000,42,,,Hai Phong | Phu Ly | Viet Tri
region,"Vùng: East, Central, West",str,0.000000,3,,,East | Central | West
district,"Quận (39 giá trị, dạng 'District #n')",str,0.000000,39,,,District #13 | District #04 | District #06


> **3 điều lạ của bảng `geography` (nhóm điền):**
> 1. …
> 2. …
> 3. …

### Từ điển bảng `payments`

- **1 dòng là:** 1 lần thanh toán của 1 đơn (mỗi đơn đúng 1 dòng)
- **Khoá chính:** order_id
- **Khoá nối:** order_id → orders

In [16]:
show_profile("payments")

cot,y_nghia,kieu,pct_trong,so_gia_tri_khac_nhau,min,max,vi_du
order_id,Mã đơn,int64,0.000000,646945,1,834397,1 | 2 | 3
payment_method,Phương thức thanh toán (giống orders.payment_method),str,0.000000,5,,,credit_card | cod | paypal
payment_value,"Số tiền thanh toán = tiền hàng net của đơn (sau chiết khấu, không gồm phí ship)",float64,0.000000,595420,389.7,3.316e+05,"7,968 | 7.116e+04 | 3.366e+04"
installments,"Số kỳ trả góp: 1, 2, 3, 6, 12",int64,0.000000,5,1,12,3 | 1 | 12


> **3 điều lạ của bảng `payments` (nhóm điền):**
> 1. …
> 2. …
> 3. …

### Từ điển bảng `shipments`

- **1 dòng là:** 1 lần giao hàng của 1 đơn (chỉ đơn đã giao/đang giao/đã trả)
- **Khoá chính:** order_id
- **Khoá nối:** order_id → orders

In [17]:
show_profile("shipments")

cot,y_nghia,kieu,pct_trong,so_gia_tri_khac_nhau,min,max,vi_du
order_id,Mã đơn,int64,0.000000,566067,1,834325,1 | 2 | 3
ship_date,Ngày xuất kho,datetime64[us],0.000000,3831,2012-07-04,2022-12-29,2012-07-07 | 2012-07-06 | 2012-07-04
delivery_date,Ngày giao tới khách (sau ship_date 2–7 ngày),datetime64[us],0.000000,3831,2012-07-06,2022-12-31,2012-07-11 | 2012-07-10 | 2012-07-07
shipping_fee,Phí vận chuyển (0–32),float64,0.000000,1856,0,32,1.37 | 2.6 | 2.38


> **3 điều lạ của bảng `shipments` (nhóm điền):**
> 1. …
> 2. …
> 3. …

### Từ điển bảng `returns`

- **1 dòng là:** 1 lần trả 1 sản phẩm của 1 đơn
- **Khoá chính:** return_id
- **Khoá nối:** (order_id, product_id) → order_items

In [18]:
show_profile("returns")

cot,y_nghia,kieu,pct_trong,so_gia_tri_khac_nhau,min,max,vi_du
return_id,Mã lượt trả hàng (RET-xxxxxx),str,0.000000,39939,,,RET-000001 | RET-000002 | RET-000003
order_id,Mã đơn,int64,0.000000,36062,2,833351,2 | 32 | 35
product_id,Mã sản phẩm bị trả,int64,0.000000,1286,3,2412,609 | 1862 | 2359
return_date,Ngày trả,datetime64[us],0.000000,3806,2012-07-11,2022-12-31,2012-07-25 | 2012-07-16 | 2012-07-11
return_reason,"Lý do: wrong_size, defective, not_as_described, changed_mind, late_delivery",str,0.000000,5,,,late_delivery | wrong_size | defective
return_quantity,Số lượng trả,int64,0.000000,8,1,8,6 | 2 | 1
refund_amount,Số tiền hoàn,float64,0.000000,39560,458.8,1.609e+05,"5.246e+04 | 5,141 | 5,316"


> **3 điều lạ của bảng `returns` (nhóm điền):**
> 1. …
> 2. …
> 3. …

### Từ điển bảng `reviews`

- **1 dòng là:** 1 đánh giá của khách cho 1 sản phẩm trong 1 đơn
- **Khoá chính:** review_id
- **Khoá nối:** order_id → orders; product_id → products; customer_id → customers

In [19]:
show_profile("reviews")

cot,y_nghia,kieu,pct_trong,so_gia_tri_khac_nhau,min,max,vi_du
review_id,Mã đánh giá (REV-xxxxxxx),str,0.000000,113551,,,REV-0000001 | REV-0000002 | REV-0000003
order_id,Mã đơn,int64,0.000000,111369,1,833296,1 | 3 | 10
product_id,Mã sản phẩm được đánh giá,int64,0.000000,1412,3,2412,2400 | 396 | 1431
customer_id,Mã khách đánh giá,int64,0.000000,48676,2,157563,58578 | 58811 | 49101
review_date,Ngày đánh giá,datetime64[us],0.000000,3825,2012-07-10,2022-12-31,2012-07-24 | 2012-08-03 | 2012-07-23
rating,Số sao 1–5,int64,0.000000,5,1,5,5 | 4 | 3
review_title,"Tiêu đề đánh giá (câu ngắn có sẵn, vd. 'Very satisfied')",str,0.000000,18,,,Highly recommend | Very satisfied | Great quality


> **3 điều lạ của bảng `reviews` (nhóm điền):**
> 1. …
> 2. …
> 3. …

### Từ điển bảng `promotions`

- **1 dòng là:** 1 chiến dịch khuyến mãi, lên lịch trước (start_date → end_date)
- **Khoá chính:** promo_id
- **Khoá nối:** promo_id ← order_items.promo_id / promo_id_2

In [20]:
show_profile("promotions")

cot,y_nghia,kieu,pct_trong,so_gia_tri_khac_nhau,min,max,vi_du
promo_id,Mã khuyến mãi (PROMO-xxxx),str,0.000000,50,,,PROMO-0001 | PROMO-0002 | PROMO-0003
promo_name,"Tên chiến dịch + năm (Spring Sale, Mid-Year Sale, Fall Launch, Year-End Sale, Urban Blowout, Rural Special)",str,0.000000,50,,,Spring Sale 2013 | Mid-Year Sale 2013 | Fall Launch 2013
promo_type,Loại: percentage (giảm %) hoặc fixed (giảm số tiền),str,0.000000,2,,,percentage | fixed
discount_value,Mức giảm (% hoặc số tiền tuỳ promo_type),float64,0.000000,6,10,50,12 | 18 | 10
start_date,Ngày bắt đầu,datetime64[us],0.000000,50,2013-01-31,2022-11-18,2013-03-18 | 2013-06-23 | 2013-08-30
end_date,Ngày kết thúc,datetime64[us],0.000000,50,2013-03-01,2022-12-31,2013-04-17 | 2013-07-22 | 2013-10-02
applicable_category,Danh mục áp dụng (trống = mọi danh mục),str,80.000000,2,,,Streetwear | Outdoor
promo_channel,"Kênh áp dụng: all_channels, online, email, social_media, in_store",str,0.000000,5,,,email | online | all_channels
stackable_flag,1 = được cộng dồn với khuyến mãi khác,int64,0.000000,2,0,1,1 | 0
min_order_value,Giá trị đơn tối thiểu (0 = không yêu cầu),int64,0.000000,5,0,200000,0 | 50000 | 150000


> **3 điều lạ của bảng `promotions` (nhóm điền):**
> 1. …
> 2. …
> 3. …

### Từ điển bảng `inventory`

- **1 dòng là:** 1 sản phẩm × 1 tháng (ảnh chụp tồn kho ngày cuối tháng)
- **Khoá chính:** (snapshot_date, product_id)
- **Khoá nối:** product_id → products

In [21]:
show_profile("inventory")

cot,y_nghia,kieu,pct_trong,so_gia_tri_khac_nhau,min,max,vi_du
snapshot_date,Ngày chụp tồn kho (ngày cuối tháng),datetime64[us],0.000000,126,2012-07-31,2022-12-31,2022-10-31 | 2022-11-30 | 2022-12-31
product_id,Mã sản phẩm,int64,0.000000,1624,1,2412,1 | 3 | 4
stock_on_hand,Tồn kho cuối tháng,int64,0.000000,1895,3,2673,3 | 35 | 36
units_received,Số lượng nhập trong tháng,int64,0.000000,360,1,817,1 | 13 | 11
units_sold,Số lượng bán trong tháng,int64,0.000000,303,1,670,1 | 11 | 10
stockout_days,Số ngày hết hàng trong tháng (0–28),int64,0.000000,29,0,28,2 | 1 | 0
days_of_supply,"Số ngày tồn kho còn đủ bán (có giá trị rất lớn, tới 68,100)",float64,0.000000,9289,5.2,6.81e+04,90 | 95.5 | 108
fill_rate,Tỷ lệ đáp ứng đơn (0–1),float64,0.000000,29,0.0667,1,0.9333 | 0.9667 | 1
stockout_flag,1 = có hết hàng trong tháng,int64,0.000000,2,0,1,1 | 0
overstock_flag,1 = tồn kho dư,int64,0.000000,2,0,1,0 | 1


> **3 điều lạ của bảng `inventory` (nhóm điền):**
> 1. …
> 2. …
> 3. …